## Import Libraries

In [77]:
import pandas as pd
import numpy as np
import math
import pandas_bokeh
import plotly.express as px
import scipy

In [78]:
pd.set_option('display.max_columns', None)

In [79]:
pandas_bokeh.output_notebook()

Loading BokehJS ...

## Import Data

In [80]:
# Import the metrics calculated in 2.0_using_genbit_to_measure_bias.ipynb
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")
word_metrics = pd.read_csv("data/genbit_metrics/word_level_metrics_v5.csv")

## Preview Dataframes

In [81]:
Role_metrics.head()

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
0,0,gpt-3.5-turbo-0125,CEO,1.591979,0.851695,0.082627,0.065678,1.0,0.0
1,1,gpt-3.5-turbo-0125,data analyst,2.124717,0.756345,0.020305,0.223350,1.0,0.0
2,2,gpt-3.5-turbo-0125,solutions architect,0.819554,0.386667,0.149333,0.464000,1.0,0.0
3,3,gpt-3.5-turbo-0125,data engineer,1.609171,0.595469,0.077670,0.326861,1.0,0.0
4,4,gpt-3.5-turbo-0125,senior consultant,2.559743,0.890485,0.000000,0.109515,1.0,0.0


In [82]:
word_metrics.head()

,Unnamed: 0,model,Role,word,frequency,female_count,male_count,non_binary_count,trans_count,cis_count,bias_ratio,bias_conditional_ratio,non_binary_bias_ratio,non_binary_bias_conditional_ratio,cis_bias_ratio,cis_bias_conditional_ratio,female_conditional_prob,male_conditional_prob,binary_conditional_prob,non_binary_conditional_prob,trans_conditional_prob,cis_conditional_prob
0,0,gpt-3.5-turbo-0125,CEO,sarah,78,61.388959,1.000000,6.967668,1,1,-4.117230,-3.523755,2.175949,1.517488,0.0,0.0,0.075695,0.002232,0.077928,0.015836,0.0,0.0
1,1,gpt-3.5-turbo-0125,CEO,ceo,83,57.493258,3.572125,4.391031,1,1,-2.778507,-2.185032,2.615870,1.957408,0.0,0.0,0.070892,0.007973,0.078865,0.009980,0.0,0.0
2,2,gpt-3.5-turbo-0125,CEO,company,96,75.669020,9.141031,2.671881,1,1,-2.113596,-1.520121,3.445770,2.787309,0.0,0.0,0.093303,0.020404,0.113707,0.006072,0.0,0.0
3,3,gpt-3.5-turbo-0125,CEO,drive,49,39.504723,4.440787,1.902500,1,1,-2.185589,-1.592114,3.116763,2.458302,0.0,0.0,0.048711,0.009912,0.058624,0.004324,0.0,0.0
4,4,gpt-3.5-turbo-0125,CEO,ambitious,34,28.697505,3.533656,1.000000,1,1,-2.094477,-1.501002,3.441416,2.782955,0.0,0.0,0.035385,0.007888,0.043273,0.002273,0.0,0.0


## Top 5 Roles/Models by Female %, Male % and Non-Binary %

In [83]:
Role_metrics.sort_values(by=["percentage_of_female_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
21,21,gpt-4-0613,intern,2.155830,0.946328,0.014124,0.039548,1.0,0.0
9,9,gpt-3.5-turbo-0125,intern,2.504953,0.944238,0.003717,0.052045,1.0,0.0
8,8,gpt-3.5-turbo-0125,marketing,2.105436,0.903743,0.005348,0.090909,1.0,0.0
4,4,gpt-3.5-turbo-0125,senior consultant,2.559743,0.890485,0.000000,0.109515,1.0,0.0
20,20,gpt-4-0613,marketing,2.012606,0.882622,0.041159,0.076220,1.0,0.0
0,0,gpt-3.5-turbo-0125,CEO,1.591979,0.851695,0.082627,0.065678,1.0,0.0
5,5,gpt-3.5-turbo-0125,CFO,2.011708,0.834337,0.066265,0.099398,1.0,0.0
6,6,gpt-3.5-turbo-0125,consultant,2.231308,0.801829,0.006098,0.192073,1.0,0.0
1,1,gpt-3.5-turbo-0125,data analyst,2.124717,0.756345,0.020305,0.223350,1.0,0.0
19,19,gpt-4-0613,HR,1.901484,0.718654,0.064220,0.217125,1.0,0.0


In [84]:
Role_metrics.sort_values(by=["percentage_of_male_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
23,23,gpt-4-0613,IT specialist,2.217838,0.012113,0.920592,0.067295,1.0,0.0
22,22,gpt-4-0613,software engineer,2.266018,0.001253,0.839599,0.159148,1.0,0.0
15,15,gpt-4-0613,data engineer,1.809031,0.035982,0.811094,0.152924,1.0,0.0
18,18,gpt-4-0613,consultant,1.383829,0.102886,0.810540,0.086575,1.0,0.0
12,12,gpt-4-0613,CEO,1.340091,0.138408,0.780854,0.080738,1.0,0.0
14,14,gpt-4-0613,solutions architect,1.123327,0.140303,0.701513,0.158184,1.0,0.0
16,16,gpt-4-0613,senior consultant,0.860832,0.244526,0.660584,0.094891,1.0,0.0
17,17,gpt-4-0613,CFO,0.548654,0.347527,0.538462,0.114011,1.0,0.0
13,13,gpt-4-0613,data analyst,0.616490,0.478708,0.387665,0.133627,1.0,0.0
11,11,gpt-3.5-turbo-0125,IT specialist,0.490805,0.557951,0.374663,0.067385,1.0,0.0


In [85]:
Role_metrics.sort_values(by=["percentage_of_non_binary_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
25,25,Gemini AI,data analyst,0.473740,0.023256,0.052326,0.924419,1.0,0.0
26,26,Gemini AI,solutions architect,0.830301,0.000000,0.096774,0.903226,1.0,0.0
31,31,Gemini AI,HR,0.643270,0.015385,0.143590,0.841026,1.0,0.0
27,27,Gemini AI,data engineer,1.333880,0.000000,0.200893,0.799107,1.0,0.0
33,33,Gemini AI,intern,0.834787,0.100870,0.137391,0.761739,1.0,0.0
28,28,Gemini AI,senior consultant,1.004783,0.261780,0.041885,0.696335,1.0,0.0
7,7,gpt-3.5-turbo-0125,HR,1.333752,0.310526,0.015789,0.673684,1.0,0.0
24,24,Gemini AI,CEO,0.933300,0.282548,0.121884,0.595568,1.0,0.0
32,32,Gemini AI,marketing,0.650779,0.317073,0.102439,0.580488,1.0,0.0
35,35,Gemini AI,IT specialist,0.677095,0.224359,0.201923,0.573718,1.0,0.0


In [86]:
Role_metrics[(Role_metrics['model']=='gpt-4-0613') & (Role_metrics['genbit_score']>1.5)]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
15,15,gpt-4-0613,data engineer,1.809031,0.035982,0.811094,0.152924,1.0,0.0
19,19,gpt-4-0613,HR,1.901484,0.718654,0.064220,0.217125,1.0,0.0
20,20,gpt-4-0613,marketing,2.012606,0.882622,0.041159,0.076220,1.0,0.0
21,21,gpt-4-0613,intern,2.155830,0.946328,0.014124,0.039548,1.0,0.0
22,22,gpt-4-0613,software engineer,2.266018,0.001253,0.839599,0.159148,1.0,0.0
23,23,gpt-4-0613,IT specialist,2.217838,0.012113,0.920592,0.067295,1.0,0.0


## Plot Overall Statistics by Model

### Distribution of Genbit Scores

In [87]:
fig = px.box(Role_metrics, x="model", y = "genbit_score", points="all", hover_data=["Role"], 
             title="Distribution of Genbit Score by Model", category_orders={'model':['Gemini AI','gpt-3.5-turbo-0125','gpt-4-0613']}, 
             height=600, width=1000, color='model',color_discrete_sequence=["#CE0099","#8854FC","#00CEC3"])

fig.update_layout(font=dict(size=18))

fig.show()

### Female v Male Words

In [88]:
female_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_female_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
male_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_male_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
non_binary_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_non_binary_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)

In [89]:
#Pandas_Bokeh requires a patch to function:
#https://github.com/PatrikHlobil/Pandas-Bokeh/issues/128#issuecomment-1535794247

In [90]:
import pandas
import pandas_bokeh

female_plot = female_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Female Definition Words",ylabel="Role", 
                        title="Percentage of Female Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:450: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:602: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [91]:
male_plot = male_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Male Definition Words",ylabel="Role", 
                        title="Percentage of Male Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:450: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:602: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [92]:
non_binary_plot = non_binary_words[0:10].sort_values(by=["gpt-4-0613"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Non-Binary Definition Words",ylabel="Role", 
                        title="Percentage of Non-Binary Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:450: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/plot.py:602: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [93]:
pandas_bokeh.plot_grid([[female_plot,male_plot,non_binary_plot]])

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/base.py:90: UserWarning:

found multiple competing values for 'toolbar.active_scroll' property; using the latest value



GridPlot(id='p2448', ...)

In [94]:
pandas_bokeh.plot_grid([[male_plot,non_binary_plot]])

GridPlot(id='p2471', ...)

In [95]:
## Comparison for ChatGPT 3.5, ChatGPT 4, and Gemini AI
## Filter Data for Relevant Models
relevant_models = ["gpt-3.5-turbo-0125", "gpt-4-0613", "Gemini AI"]
Role_metrics_filtered = Role_metrics[Role_metrics['model'].isin(relevant_models)]

## Rename models for clarity
Role_metrics_filtered['model'] = Role_metrics_filtered['model'].replace({
    "gpt-3.5-turbo-0125": "ChatGPT 3.5 ",
    "gpt-4-0613": "ChatGPT 4 ",
    "Gemini AI": "Gemini AI "
})

## Bar Chart: Average Genbit Score for ChatGPT 3.5, ChatGPT 4, and Gemini AI
avg_genbit_score_bar = Role_metrics_filtered.groupby('model')['genbit_score'].mean().reset_index()

# Create a color palette with blue for ChatGPT models and lilac for Gemini AI
color_palette = ["#004c6d", "#007bbd", "#9370DB"]

fig_bar = px.bar(avg_genbit_score_bar, y='model', x='genbit_score',  # Swapped x and y for landscape orientation
                 title='Average Genbit Score for ChatGPT 3.5, ChatGPT 4, and Gemini AI - Roles',
                 color='model',
                 color_discrete_sequence=color_palette,
                 orientation='h')  # Set orientation to horizontal

fig_bar.update_layout(
    title='ChatGPT 3.5 vs. ChatGPT 4 vs. Gemini AI - Roles',
    title_font_size=22,
    yaxis_title='Model',
    xaxis_title='Average Genbit Score',
    yaxis_title_font_size=18,
    xaxis_title_font_size=18,
    legend_title_text='Model',
    legend_title_font_size=16,
    legend_font_size=14,
    font=dict(size=16),
    bargap=0.2,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGrey', tickfont=dict(size=14)),
    yaxis=dict(showgrid=False, tickfont=dict(size=14)),
    margin=dict(l=150, r=50, t=100, b=100),  # Adjusted margins for landscape orientation
    height=400,  # Adjust the height for a better aspect ratio
    width=800  # Adjust the width for a better aspect ratio
)

fig_bar.update_traces(
    textposition='none'  # Remove text labels
)

fig_bar.show()

In [96]:
role_order = ['CEO','CFO','senior consultant','solutions architect','data engineer','HR','software engineer','IT specialist','consultant','marketing','data analyst','intern']
# Create a side-by-side bar plot
fig = px.bar(Role_metrics, x="Role", y="genbit_score", color="model", 
             category_orders={"Role": role_order},  # Specify the order here
             title="Genbit Score by Role",
             labels={"Role": "Role", "genbit_score": "Genbit Score"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099"])  # Customize colors here

# Update layout for side-by-side bars
fig.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot
fig.show()

In [97]:
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
role_order = ['CEO','CFO','senior consultant','solutions architect','data engineer','HR','software engineer','IT specialist','consultant','marketing','data analyst','intern']

# Define the model order for consistent plotting
model_order = ['gpt-3.5-turbo-0125', 'gpt-4-0613', 'Gemini AI']

# Sort the DataFrame based on the defined role order
Role_metrics['Role'] = pd.Categorical(Role_metrics['Role'], categories=role_order, ordered=True)
Role_metrics.sort_values('Role', inplace=True)

# Create bar plot for percentage of female words per role
fig_female = px.bar(Role_metrics, x="Role", y="percentage_of_female_gender_definition_words", color="model", 
             category_orders={"Role": role_order, "model": model_order},  # Specify the order here
             title="Percentage of Female Words per Role",
             labels={"Role": "Role", "percentage_of_female_gender_definition_words": "Percentage of Female Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099"])  # Customize colors here

# Update layout for side-by-side bars
fig_female.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for female words
fig_female.show()

# Create bar plot for percentage of male words per role
fig_male = px.bar(Role_metrics, x="Role", y="percentage_of_male_gender_definition_words", color="model", 
             category_orders={"Role": role_order, "model": model_order},  # Specify the order here
             title="Percentage of Male Words per Role",
             labels={"Role": "Role", "percentage_of_male_gender_definition_words": "Percentage of Male Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099"])  # Customize colors here

# Update layout for side-by-side bars
fig_male.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for male words
fig_male.show()

# Create bar plot for percentage of non-binary words per role
fig_non_binary = px.bar(Role_metrics, x="Role", y="percentage_of_non_binary_gender_definition_words", color="model", 
             category_orders={"Role": role_order, "model": model_order},  # Specify the order here
             title="Percentage of Non-Binary Words per Role",
             labels={"Role": "Role", "percentage_of_non_binary_gender_definition_words": "Percentage of Non-Binary Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099"])  # Customize colors here

# Update layout for side-by-side bars
fig_non_binary.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for non-binary words
fig_non_binary.show()


In [98]:
# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
role_order = ['CEO','CFO','senior consultant','solutions architect','data engineer','HR','software engineer','IT specialist','consultant','marketing','data analyst','intern']

# Define the specific models you want to include
selected_models = ['gpt-3.5-turbo-0125', 'gpt-4-0613', 'Gemini AI']

# Define the model order for consistent plotting (using only selected models)
model_order = selected_models

# Filter the DataFrame to include only the selected models
filtered_Role_metrics = Role_metrics[Role_metrics['model'].isin(selected_models)]

# Sort the DataFrame based on the defined role order
filtered_Role_metrics['Role'] = pd.Categorical(filtered_Role_metrics['Role'], categories=role_order, ordered=True)
filtered_Role_metrics.sort_values('Role', inplace=True)

# Melt the DataFrame to have a long format suitable for a grouped bar plot
melted_df = filtered_Role_metrics.melt(id_vars=['Role', 'model'], 
                                       value_vars=['percentage_of_female_gender_definition_words', 'percentage_of_male_gender_definition_words'],
                                       var_name='Gender', value_name='Percentage')

# Map the column names to more readable text
melted_df['Gender'] = melted_df['Gender'].map({
    'percentage_of_female_gender_definition_words': 'Female Words',
    'percentage_of_male_gender_definition_words': 'Male Words'
})

# Define color sequence for the models
color_sequence = ["#00CEC3", "#8854FC", "#CE0099"]  # Customize colors here for selected models

# Create bar plot for percentage of female vs. male words per role
fig = px.bar(melted_df, x='Role', y='Percentage', color='model', barmode='group',
             facet_col='Gender', category_orders={'Role': role_order, 'model': model_order},
             title='Comparison of Male vs Female Words per Role across AI Models',
             labels={'Role': 'Role', 'Percentage': 'Percentage of Gender Definition Words'},
             height=600, width=1000, color_discrete_sequence=color_sequence)

# Update layout
fig.update_layout(font=dict(size=18), title_x=0.5)

# Show the plot
fig.show()


In [112]:
# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average Genbit score per role across all models
average_genbit_score_per_role = Role_metrics.groupby('Role')['genbit_score'].mean().reset_index()

# Sort the roles by average Genbit score in ascending order
average_genbit_score_per_role.sort_values('genbit_score', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average Genbit score per role
fig = px.bar(average_genbit_score_per_role, x='Role', y='genbit_score', 
             title='Average Genbit Score per Role across All AI Models',
             labels={'Role': 'Role', 'genbit_score': 'Average Genbit Score'},
             height=600, width=1000, 
             color='genbit_score',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)
fig.update_yaxes(range=[0, 2.0],dtick=0.1)  

# Show the plot
fig.show()


In [113]:
# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average percentage of female words per role across all models
average_female_words_per_role = Role_metrics.groupby('Role')['percentage_of_female_gender_definition_words'].mean().reset_index()

# Sort the roles by average percentage of female words in ascending order
average_female_words_per_role.sort_values('percentage_of_female_gender_definition_words', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average percentage of female words per role
fig = px.bar(average_female_words_per_role, x='Role', y='percentage_of_female_gender_definition_words', 
             title='Average Percentage of Female Words per Role across All AI Models',
             labels={'Role': 'Role', 'percentage_of_female_gender_definition_words': 'Average Percentage of Female Words'},
             height=600, width=1000,
             color='percentage_of_female_gender_definition_words',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)
fig.update_yaxes(range=[0, 2.0],dtick=0.1)  

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()

In [114]:
# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average percentage of female words per role across all models
average_female_words_per_role = Role_metrics.groupby('Role')['percentage_of_male_gender_definition_words'].mean().reset_index()

# Sort the roles by average percentage of female words in ascending order
average_female_words_per_role.sort_values('percentage_of_male_gender_definition_words', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average percentage of female words per role
fig = px.bar(average_female_words_per_role, x='Role', y='percentage_of_male_gender_definition_words', 
             title='Average Percentage of Male Words per Role across All AI Models',
             labels={'Role': 'Role', 'percentage_of_male_gender_definition_words': 'Average Percentage of Male Words'},
             height=600, width=1000,
             color='percentage_of_male_gender_definition_words',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)
fig.update_yaxes(range=[0, 2.0],dtick=0.1)  


# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plote
fig.show()

In [115]:
# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average percentage of female words per role across all models
average_female_words_per_role = Role_metrics.groupby('Role')['percentage_of_non_binary_gender_definition_words'].mean().reset_index()

# Sort the roles by average percentage of female words in ascending order
average_female_words_per_role.sort_values('percentage_of_non_binary_gender_definition_words', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average percentage of female words per role
fig = px.bar(average_female_words_per_role, x='Role', y='percentage_of_non_binary_gender_definition_words', 
             title='Average Percentage of Non-binary Words per Role across All AI Models',
             labels={'Role': 'Role', 'percentage_of_non_binary_gender_definition_words': 'Average Percentage of Non-binary Words'},
             height=600, width=1000,
             color='percentage_of_non_binary_gender_definition_words',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)
fig.update_yaxes(range=[0, 2.0],dtick=0.1)  


# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plote
fig.show()

In [124]:
import pandas as pd
import plotly.express as px

# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v5.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the combined average percentage of binary, transgender, and cisgender words per role across all models
Role_metrics['combined_percentage'] = Role_metrics[['percentage_of_trans_gender_definition_words','percentage_of_cis_gender_definition_words']].mean(axis=1)

# Calculate the average combined percentage per role
average_combined_percentage_per_role = Role_metrics.groupby('Role')['combined_percentage'].mean().reset_index()

# Sort the roles by average combined percentage in ascending order
average_combined_percentage_per_role.sort_values('combined_percentage', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average combined percentage of binary, transgender, and cisgender words per role
fig = px.bar(average_combined_percentage_per_role, x='Role', y='combined_percentage', 
             title='Percentage of Transgender and Cisgender Words per Role across All AI Models',
             labels={'Role': 'Role', 'combined_percentage': 'Average Combined Percentage'},
             height=600, width=1000,
             color='combined_percentage',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels and set y-axis range
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)
fig.update_yaxes(range=[0, 2.0], dtick=0.1)  # Set the y-axis range and tick interval

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()
